In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import requests

def hava_durumu(sehir:str) -> str:
    r = requests.get(f"https://wttr.in/{sehir}?format=3&lang=tr", timeout= 10)
    return r.text.strip()

def hesapla(ifade: str) -> str:
    # eval öğretim amaçlı.Üretimde güvenli bir parser kullan. (örn: ast/numexpr).
    return str(eval(ifade))

ARACLAR = {"hava_durumu": hava_durumu, "hesapla": hesapla}

print(hava_durumu("Burdur"))
print(hesapla("184 * 27"))

Burdur: ☁️  +10°C
4968


# OpenAI - Tool Şemaları (JSON Schema)

Her tool: name (fonksiyon adı) + description (model bunu okuyup ne zaman cağrılacağına karar verir) + parameters (argümanlar, JSON Schema). 


OpenAI'da her tool {"type": "function", "function: {...}} ile sarılı, argünan şeması parameters anahtarındadır.

In [ ]:
openai_tools = [
    {
        "type": "functions",
        "functions": {
            "name":"hava_durumu",
            "description": "Bir şehir güncel hava durumunu getirir",
            "parameters:": {
                "type": "object",
                "properties": {"sehir": {"type": "string"}},
                "required": ["sehir"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "hesapla",
            "description": "Bir aritmetik ifadeyi hesaplar, örn '184 * 27'",
            "parameters": {
                "type": "object",
                "properties": {"ifade": {"type","string"}},
                "required": ["ifade"],
            },
        },
    },
]

In [5]:
from openai import OpenAI
import json

client = OpenAI()

messages = [
    {
        "role": "user",
        "content": "İstanbul'da hava nasıl? Bir de 184 + 27 kaç eder?"
    }
]

while True:
    resp = client.chat.completions.create(
        model = "gpt-5.4-mini",
        messages= messages,
        tools= openai_tools   
    )
    msg = resp.choices[0].message
    messages.append(msg)

    if not msg.tool_calls:
        print("Nihai Cevap:", msg.content)
        break


    for tc in msg.tool_calls:
        ad = tc.function.name
        args = json.loads(tc.function.arguments)
        sonuc = ARACLAR[ad](**args)
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": str(sonuc)
        })

        


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [1]:
antropic_tools = [
    {
        "name": "hava_durumu",
            "description": "Bir şehrin güncel hava durumunu getirir",
            "input_shape": {
                "type": "object",
                "properties": {"sehir": {"type": "string"}},
                "required": ["sehir"],
        },
    },
    {
        "name":"hesapla",
        "description": "Bir aritmetik ifadeyi hesaplar, örn: '184 * 27'",
        "input_schema": {
            "type": "object",
            "properties": {"ifade": {"type": "string"}},
            "required": ["ifade"],
        },
    },
]

In [ ]:
from anthropic import Antropic

client = Antropic()

messages = [
    {"role": "user",
    "content": "İstanbul'da hava nasıl? Bir de 184 * 27 kaç eder?"}
]

resp = client.messages.create(
    model = "claude-haiku-4-5",
    max_tokens = 1024,
    tools = antropic_tools,
    messages = messages
)

while resp.stop_reason == "tool_use":
    messages.append({"role": "assistant","content": resp.content})


    tool_results = []
    for block in resp.content:
        if block.type == "tool_use":
            ad = block.name
            print(f"-> Model çağırılıyor: {ad}({block.input})")
            sonuc = ARACLAR[ad](**block.input)
            tool_results.append({
                "type":"tool_result",
                "tool_use_id": block.id,
                "content": str(sonuc)
            })


    messages.append({"role": "user", "content":tool_results})

    resp = client.messages.create(
        model = "claude-haiku-4.5",
        max_tokens = 1024,
        tools = antropic_tools,
        messages = messages
    )

print("Nihai Cevap", resp.content[0].text)

In [ ]:
ucus_tool = {
    "type": "function",
    "function": {
        "name": "ucus_ara",
        "description": "İki şehir arası uçuş arar",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "kalkis": {"type":"string"},
                "varis": {"type": "string"},
                "yolcu": {"type": "integer"},
            },
            "required": ["kalkis","varis","yolcu"],
            "additionalProperties": False,
        },
    },
}

# Beklenen davranış: model "yolcu" alanını her zaman integer olarak üretir.
# şemada tanımsız bir alan (ör. 'tarih') eklemeye çalışmaz.

In [ ]:
# LangChain 

import requests
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def hava_durumu(sehir: str) -> str:
    """Bir şehrin güncel hava durumunu getirir"""
    return requests.get(f"https://wttr.in/{sehir}?format=3&lang=tr", timeout= 10).text.strip()

@tool
def hesapla(ifade: str) -> str:
    return str(eval(ifade))

llm = ChatOpenAI(model = "gpt-5.5-mini", temperature= 0)
llm_with_tools = llm.bind_tools([hava_durumu, hesapla])

cevap = llm_with_tools.invoke("İstanbul'da hava nasıl?")

print(cevap.tool_calls)


In [ ]:
from langchain_core.messages import ToolMessage

araclar = {"hava_durumu": hava_durumu, "hesapla": hesapla}
mesajlar = [("user,", "İstanbul'da hava nasıl? Bir de 184*27 kaç eder?")]

cevap = llm_with_tools.invoke(mesajlar)
mesajlar.append(cevap)

for tc in cevap.tool_calls:
    sonuc = araclar[tc["name"]].invoke(tc["args"])

    mesajlar.append(ToolMessage(sonuc, tool_call_id = tc["id"]))

final = llm_with_tools.invoke(mesajlar)
print(final.content)